# 05_sensitivity_analyses

This notebook runs the supplementary sensitivity analyses used to evaluate robustness of the primary findings.

## Purpose

The workflow reruns the primary pharmacokinetic reconstruction or regression framework under alternate model specifications, cohort definitions, and covariate structures. Outputs are used for supplementary figures and tables.

## Main sensitivity analyses

1. Alternative pharmacokinetic model using the Schnider framework
2. Inclusion of the administratively masked 90+ cohort
3. Unadjusted regression models
4. Broader-input cohort with relaxed preprocessing assumptions
5. Adjustment for co-administered induction medications

## Input datasets

This notebook assumes availability of patient-level simulation outputs generated in prior workflow steps. Depending on the sensitivity analysis being run, the input file should correspond to the appropriate Eleveld-based, Schnider-based, broader-input, or medication-adjusted patient-level dataset.

## Main outputs

- supplementary sensitivity-analysis figures
- age-stratified sensitivity-analysis tables
- combined supplementary master sensitivity table

## Load analysis dataset

Set `file_path` to the appropriate patient-level simulation dataset for the sensitivity analysis being run in this section.

Examples include:
- the primary Eleveld-derived dataset for unadjusted analyses
- the Schnider-derived dataset for PK-model sensitivity analysis
- the broader-input Eleveld dataset for relaxed-preprocessing sensitivity analysis
- the medication-adjusted Eleveld dataset for co-medication sensitivity analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random
import statsmodels.api as sm
import statsmodels.formula.api as smf
from patsy import bs



## Sensitivity analysis 1: Schnider pharmacokinetic model

In [ ]:
# --- 3. LOAD THE DATA ---
file_path = "INSERT_INPUT_FILE_HERE.pkl"  # e.g., primary Eleveld, Schnider, relaxed-input, or medication-adjusted dataset

if os.path.exists(file_path):
    df_analysis = pd.read_pickle(file_path)
    print(f"Loaded {len(df_analysis):,} rows from {file_path}.")
else:
    raise FileNotFoundError(
        f"Input file not found: {file_path}. "
        "Update file_path to the appropriate patient-level simulation dataset before running this section."
    )

In [ ]:


# Initial Cleaning
df_reg = df_analysis.copy()
df_reg["AGE"] = pd.to_numeric(df_reg["AGE"], errors="coerce")
df_reg = df_reg[(df_reg["AGE"] >= 18) & (df_reg["AGE"] < 90)].copy()

# Standardize Covariates
if df_reg["SEX"].dtype == "object":
    df_reg["SEX"] = df_reg["SEX"].str.upper().map({"M": 0, "F": 1})

# Define Bins & Labels
bins = [18, 25, 35, 45, 55, 65, 75, 85, 90]
labels = ["18-24", "25-34", "35-44", "45-54", "55-64", "65-74", "75-84", "85-89"]
df_reg["Age_Group"] = pd.cut(df_reg["AGE"], bins=bins, labels=labels, right=False)

# Remove missing data to ensure the model fits
df_reg = df_reg.dropna(subset=["Max_Effect_Concentration", "ASA_SCORE", "BMI", "SEX"]).copy()

# --- 2. MODELING (Schneider Analysis) ---
# ---------------------------------------------------------------------------
AGE_LO, AGE_HI = float(df_reg["AGE"].min()), float(df_reg["AGE"].max())

# Fit Spline Model
formula = f"Max_Effect_Concentration ~ bs(AGE, df=5, lower_bound={AGE_LO}, upper_bound={AGE_HI}) + ASA_SCORE + BMI + SEX"
model_sch = smf.ols(formula, data=df_reg).fit()

# Standardization Means
g_asa, g_bmi, g_sex = df_reg[["ASA_SCORE", "BMI", "SEX"]].mean()

# Predictions for the Continuous Trend Line
age_grid = np.linspace(18, 89, 400)
grid_df = pd.DataFrame({"AGE": age_grid, "ASA_SCORE": g_asa, "BMI": g_bmi, "SEX": g_sex})
pred_grid = model_sch.get_prediction(grid_df).summary_frame()

# Predictions for the Binned Bars
bin_stats = df_reg.groupby("Age_Group", observed=True).agg(mean_age=("AGE", "mean")).reset_index()
bin_pred_input = bin_stats.rename(columns={"mean_age": "AGE"}).assign(ASA_SCORE=g_asa, BMI=g_bmi, SEX=g_sex)
bin_pred_ce = model_sch.get_prediction(bin_pred_input).summary_frame()

# --- 3. VISUALIZATION (Indigo / Steel Palette) ---
# ---------------------------------------------------------------------------
plt.figure(figsize=(15, 8.5))

# A. The Adjusted Spline (Indigo)
plt.plot(age_grid, pred_grid["mean"], linewidth=4, color='#3f51b5',
         label="Adj. Predicted Peak $C_e$ (Schneider Model)")

# B. 95% Confidence Interval Band
plt.fill_between(age_grid, pred_grid["mean_ci_lower"], pred_grid["mean_ci_upper"],
                 color='#3f51b5', alpha=0.15)

# C. Binned Bars (Steel Blue)
plt.bar(bin_stats["mean_age"], bin_pred_ce["mean"], width=4.5, color='#7986cb',
        alpha=0.6, edgecolor='#3f51b5',
        yerr=[bin_pred_ce["mean"] - bin_pred_ce["mean_ci_lower"],
              bin_pred_ce["mean_ci_upper"] - bin_pred_ce["mean"]],
        capsize=6, label="Binned Cohort Means")

# D. Cohort Value Labels
for i, row in bin_pred_ce.iterrows():
    plt.text(bin_stats["mean_age"].iloc[i], row["mean"] + 0.05,
             f'{row["mean"]:.2f}', ha='center', fontsize=11,
             fontweight='bold', color='#3f51b5')

# Dynamics & Cosmetics
current_max = bin_pred_ce["mean_ci_upper"].max()
plt.ylim(0, current_max * 1.4)

plt.xticks(bin_stats["mean_age"], labels, fontsize=10)
plt.ylabel("Effect-Site Concentration (mcg/mL)", fontsize=12)
plt.xlabel("Age Cohort (Years)", fontsize=12)
plt.title("Sensitivity Analysis: Predicted Peak Exposure using Schneider Model Parameters\n(Standardized to Mean BMI/ASA/Sex)",
          fontsize=15, fontweight='bold', pad=30)
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.legend(loc="upper right", frameon=True)
plt.tight_layout()

plt.show()

In [ ]:
# --- MASTER SENSITIVITY FIGURE: SCHNIDER MODEL (UPDATED LABELS) ---
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

# 1. CALCULATIONS & PERCENTAGE MAPPING
# ---------------------------------------------------------------------------
# Modeling Dose
formula_dose = f"DOSE_PER_ABW ~ bs(AGE, df=5, lower_bound={AGE_LO}, upper_bound={AGE_HI}) + ASA_SCORE + BMI + SEX"
model_dose = smf.ols(formula_dose, data=df_reg).fit()

# Predictions
pred_grid_ce = model_sch.get_prediction(grid_df).summary_frame()
pred_grid_dose = model_dose.get_prediction(grid_df).summary_frame()

bin_pred_ce = model_sch.get_prediction(bin_pred_input).summary_frame()
bin_pred_dose = model_dose.get_prediction(bin_pred_input).summary_frame()

# Normalization to first bin (18-24)
dose_baseline = bin_pred_dose["mean"].iloc[0]
ce_baseline = bin_pred_ce["mean"].iloc[0]

bin_stats["Dose_Pct"] = (bin_pred_dose["mean"] / dose_baseline) * 100
bin_stats["Ce_Pct"] = (bin_pred_ce["mean"] / ce_baseline) * 100
bin_stats["Ce_L"] = (bin_pred_ce["mean_ci_lower"] / ce_baseline) * 100
bin_stats["Ce_U"] = (bin_pred_ce["mean_ci_upper"] / ce_baseline) * 100

# 2. FIGURE SETUP (Height=19, hspace=0.32)
# ---------------------------------------------------------------------------
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 19))
plt.subplots_adjust(hspace=0.32, top=0.95, bottom=0.08, left=0.1, right=0.95)

# --- PANEL A: SCHNIDER EXPOSURE ---
ax1.plot(age_grid, pred_grid_ce["mean"], linewidth=4, color='#3f51b5', label="Adj. Predicted Peak Ce (Schnider)")
ax1.bar(bin_stats["mean_age"], bin_pred_ce["mean"], width=3.5, color='#7986cb', alpha=0.6, edgecolor='#3f51b5',
        yerr=[bin_pred_ce["mean"] - bin_pred_ce["mean_ci_lower"], bin_pred_ce["mean_ci_upper"] - bin_pred_ce["mean"]], capsize=5)

for i, row in bin_pred_ce.iterrows():
    ax1.text(bin_stats["mean_age"].iloc[i], row["mean"] + 0.08, f'{row["mean"]:.2f}', ha='center', fontweight='bold', color='#3f51b5')

ax1.set_title("A: Standardized Peak Exposure (Ce) - Schnider Model", loc='left', fontsize=14, fontweight='bold')
ax1.set_ylabel("Effect-Site Concentration (mcg/mL)")
ax1.set_ylim(0, max(bin_pred_ce["mean_ci_upper"]) * 1.3)
ax1.set_xticks(bin_stats["mean_age"])
ax1.set_xticklabels(labels)
ax1.legend(loc="upper right", frameon=True)

# --- PANEL B: TITRATION (mg/kg ABW) ---
ax2.plot(age_grid, pred_grid_dose["mean"], linewidth=4, color='#2ca02c', label="Adj. Induction Dose")
ax2.bar(bin_stats["mean_age"], bin_pred_dose["mean"], width=3.5, color='#a1d99b', alpha=0.6, edgecolor='#2ca02c',
        yerr=[bin_pred_dose["mean"] - bin_pred_dose["mean_ci_lower"], bin_pred_dose["mean_ci_upper"] - bin_pred_dose["mean"]], capsize=5)

for i, row in bin_pred_dose.iterrows():
    ax2.text(bin_stats["mean_age"].iloc[i], row["mean"] + 0.08, f'{row["mean"]:.2f}', ha='center', fontweight='bold', color='#2ca02c')

ax2.set_title("B: Standardized Clinician Induction Dose Titration", loc='left', fontsize=14, fontweight='bold')
ax2.set_ylabel("Induction Dose (mg/kg ABW)")
ax2.set_xticks(bin_stats["mean_age"])
ax2.set_xticklabels(labels)
ax2.legend(loc="upper right", frameon=True)

# --- PANEL C: RELATIVE DIVERGENCE (UPDATED LABELS) ---
x_indices = np.arange(len(labels))
ax3.plot(x_indices, bin_stats["Dose_Pct"], marker='o', linewidth=4, color='#2ca02c',
         label="Clinician Titration (mg/kg % Baseline)")
ax3.plot(x_indices, bin_stats["Ce_Pct"], marker='s', linewidth=4, color='#1f77b4',
         label="Modeled Peak Exposure (Ce % Baseline)")
ax3.fill_between(x_indices, bin_stats["Ce_L"], bin_stats["Ce_U"], color='#1f77b4', alpha=0.15)

ax3.set_title("C: Relative Maintenance of Exposure: Percentage of Young-Adult Baseline", loc='left', fontsize=14, fontweight='bold')
ax3.set_ylabel("Percentage of Baseline (%)")
ax3.set_xlabel("Age Cohort (Years)", fontsize=13, labelpad=10)
ax3.set_xticks(x_indices)
ax3.set_xticklabels(labels)
ax3.set_ylim(40, 115)
ax3.axhline(100, color='black', linestyle=':', alpha=0.5)
ax3.legend(loc='lower left', frameon=True)
ax3.grid(axis='y', linestyle='--', alpha=0.3)

plt.savefig("figure_S1_schnider_composite.pdf", bbox_inches='tight', dpi=300)
plt.show()

# 2. SAVE TABLE S1 (UPDATED TO INCLUDE MG/KG)
table_sch = pd.DataFrame({
    'Age Group': labels,
    'N': [f"{int(x):,}" for x in df_reg.groupby("Age_Group", observed=True).size().values],

    # NEW COLUMN: Induction Dose (mg/kg)
    'Induction Dose': [f"{m:.2f} ({l:.2f}-{u:.2f})" for m,l,u in zip(bin_pred_dose['mean'], bin_pred_dose['mean_ci_lower'], bin_pred_dose['mean_ci_upper'])],

    'Peak Ce': [f"{m:.2f} ({l:.2f}-{u:.2f})" for m,l,u in zip(bin_pred_ce['mean'], bin_pred_ce['mean_ci_lower'], bin_pred_ce['mean_ci_upper'])],
    'Exposure %': [f"{m:.1f}% ({l:.1f}-{u:.1f}%)" for m,l,u in zip(bin_stats["Ce_Pct"], bin_stats["Ce_L"], bin_stats["Ce_U"])]
})

# Save to CSV (optional, helps with the master merge later)
table_sch.to_csv("table_S1_schnider.csv", index=False)

## Sensitivity analysis 2: Inclusion of the administratively masked 90+ cohort

In [ ]:
# --- 3. LOAD THE DATA ---
file_path = "INSERT_INPUT_FILE_HERE.pkl"  # e.g., primary Eleveld, Schnider, relaxed-input, or medication-adjusted dataset

if os.path.exists(file_path):
    df_analysis = pd.read_pickle(file_path)
    print(f"Loaded {len(df_analysis):,} rows from {file_path}.")
else:
    raise FileNotFoundError(
        f"Input file not found: {file_path}. "
        "Update file_path to the appropriate patient-level simulation dataset before running this section."
    )

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:

# Eleveld Target Formula
def ce50_eleveld(age):
    return 3.08 * np.exp(-0.00635 * (age - 35))


# ----------------------------
df = df_analysis.copy()
df["AGE"] = pd.to_numeric(df["AGE"], errors="coerce")


if df["SEX"].dtype == "object":
    df["SEX"] = df["SEX"].map({"M": 0, "F": 1, "m": 0, "f": 1})

bins   = [18, 25, 35, 45, 55, 65, 75, 85, 91]
labels = ["18-24", "25-34", "35-44", "45-54", "55-64", "65-74", "75-84", "85 +"]
df["Age_Group"] = pd.cut(df["AGE"], bins=bins, labels=labels, right=False)

df_reg = df.dropna(subset=["Max_Effect_Concentration", "AGE", "ASA_SCORE", "BMI", "SEX"]).copy()

# 2) MODELING
# ------------
AGE_LO, AGE_HI = float(df_reg["AGE"].min()), float(df_reg["AGE"].max())
formula_ce = f"Max_Effect_Concentration ~ bs(AGE, df=5, lower_bound={AGE_LO}, upper_bound={AGE_HI}) + ASA_SCORE + BMI + SEX"
model_ce = smf.ols(formula_ce, data=df_reg).fit()

g_asa, g_bmi, g_sex = df_reg[["ASA_SCORE", "BMI", "SEX"]].mean()

# Predictions
age_grid = np.linspace(18, 89, 400)
grid_df = pd.DataFrame({"AGE": age_grid, "ASA_SCORE": g_asa, "BMI": g_bmi, "SEX": g_sex})
pred_grid = model_ce.get_prediction(grid_df).summary_frame()

bin_stats = df_reg.groupby("Age_Group", observed=True).agg(mean_age=("AGE", "mean")).reset_index()
bin_pred_ce = model_ce.get_prediction(bin_stats.rename(columns={"mean_age": "AGE"}).assign(ASA_SCORE=g_asa, BMI=g_bmi, SEX=g_sex)).summary_frame()

# 3) VISUALIZATION: EXPOSURE
# ----------------------------
plt.figure(figsize=(15, 8.5)) # Increased height slightly
plt.plot(age_grid, pred_grid["mean"], linewidth=4, color='#1f77b4', label="Adj. Predicted Peak $C_e$")
plt.plot(age_grid, ce50_eleveld(age_grid), linestyle="--", linewidth=3, color='red', label="Eleveld $Ce_{50}$ Target")

plt.bar(bin_stats["mean_age"], bin_pred_ce["mean"], width=3.5, color='#5dade2', alpha=0.6, edgecolor='#1f77b4',
        yerr=[bin_pred_ce["mean"] - bin_pred_ce["mean_ci_lower"], bin_pred_ce["mean_ci_upper"] - bin_pred_ce["mean"]], capsize=5)

for i, row in bin_pred_ce.iterrows():
    target = ce50_eleveld(bin_stats["mean_age"].iloc[i])
    diff = ((row["mean"] / target) - 1) * 100
    # Mean Value Label
    plt.text(bin_stats["mean_age"].iloc[i], row["mean"] + 0.12, f'{row["mean"]:.2f}', ha='center', fontsize=11, fontweight='bold', color='#1f77b4')
    # Surplus Percentage Label (Higher padding)
    plt.text(bin_stats["mean_age"].iloc[i], row["mean"] + 0.38, f'+{diff:.0f}%', ha='center', fontsize=11, fontweight='bold', color='red')

plt.xticks(bin_stats["mean_age"], labels)
plt.title("Standardized Peak Exposure ($C_e$) vs. Validated Target\n", fontsize=15, fontweight='bold', pad=40)
plt.ylabel("Effect-Site Concentration (mcg/mL)")
plt.ylim(0, 5.0) # Explicitly set headroom to prevent label clipping
plt.legend(bbox_to_anchor=(1.04, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# 1) MODELING DOSE
# -----------------
formula_dose = f"DOSE_PER_ABW ~ bs(AGE, df=5, lower_bound={AGE_LO}, upper_bound={AGE_HI}) + ASA_SCORE + BMI + SEX"
model_dose = smf.ols(formula_dose, data=df_reg).fit()

pred_grid_dose = model_dose.get_prediction(grid_df).summary_frame()
bin_pred_dose = model_dose.get_prediction(bin_stats.rename(columns={"mean_age": "AGE"}).assign(ASA_SCORE=g_asa, BMI=g_bmi, SEX=g_sex)).summary_frame()

# 2) VISUALIZATION: TITRATION (No Reference Lines)
# ----------------------------
plt.figure(figsize=(15, 8))
plt.plot(age_grid, pred_grid_dose["mean"], linewidth=4, color='#2ca02c', label="Adj. Induction Dose")
plt.bar(bin_stats["mean_age"], bin_pred_dose["mean"], width=3.5, color='#a1d99b', alpha=0.6, edgecolor='#2ca02c',
        yerr=[bin_pred_dose["mean"] - bin_pred_dose["mean_ci_lower"], bin_pred_dose["mean_ci_upper"] - bin_pred_dose["mean"]], capsize=5)

for i, row in bin_pred_dose.iterrows():
    plt.text(bin_stats["mean_age"].iloc[i], row["mean"] + 0.08, f'{row["mean"]:.2f}', ha='center', fontsize=11, fontweight='bold', color='#2ca02c')

plt.xticks(bin_stats["mean_age"], labels)
plt.title("Standardized Clinician Induction Dose Titration (Ages 18-89)\n", fontsize=15, fontweight='bold', pad=20)
plt.ylabel("Induction Dose (mg/kg ABW)")
plt.ylim(0, max(bin_pred_dose["mean_ci_upper"]) + 0.5) # Dynamic headroom
plt.legend(bbox_to_anchor=(1.04, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# --- 3) FIGURE C: RELATIVE DIVERGENCE (Clean Version) ---
# ---------------------------------------------------------------------------

# 1. Calculate Baselines (from the 18-24 cohort)
b_ce     = bin_pred_ce['mean'].iloc[0]
b_dose   = bin_pred_dose['mean'].iloc[0]
b_target = ce50_eleveld(bin_stats["mean_age"].iloc[0])

# 2. Normalize cohort data to % of Baseline
bin_stats["Dose_Pct"]   = (bin_pred_dose['mean'] / b_dose) * 100
bin_stats["Ce_Pct"]     = (bin_pred_ce['mean'] / b_ce) * 100
bin_stats["Target_Pct"] = (ce50_eleveld(bin_stats["mean_age"]) / b_target) * 100

# Normalize Confidence Intervals for the shaded blue band
bin_stats["Ce_L"] = (bin_pred_ce['mean_ci_lower'] / b_ce) * 100
bin_stats["Ce_U"] = (bin_pred_ce['mean_ci_upper'] / b_ce) * 100

# 3. Visualization
plt.figure(figsize=(15, 8))
x_axis = np.arange(len(labels))

# Line 1: Clinician Dose Decay (Green)
plt.plot(x_axis, bin_stats["Dose_Pct"], marker='o', linewidth=4, color='#2ca02c',
         markersize=10, label="Clinician Titration (Induction mg/kg % Baseline)")

# Line 2: Resulting Brain Exposure (Blue)
plt.plot(x_axis, bin_stats["Ce_Pct"], marker='s', linewidth=4, color='#1f77b4',
         markersize=10, label="Predicted Peak Exposure (Ce % Baseline)")

# Blue Shaded Confidence Interval Band
plt.fill_between(x_axis, bin_stats["Ce_L"], bin_stats["Ce_U"], color='#1f77b4', alpha=0.1)

# Line 3: Physiological Target (Red Dashed)
plt.plot(x_axis, bin_stats["Target_Pct"], marker='D', linewidth=3, linestyle='--', color='red',
         markersize=8, label="Physiological Target (Ce50 % Baseline)")

# Formatting
plt.xticks(x_axis, labels)
plt.ylabel("Percentage of Young Adult (18-24) Baseline (%)", fontsize=12)
plt.title("Standardized Relative Divergence: Dose Titration vs. Physiological Requirement\n(Standardized to Mean ASA, BMI, and Sex)",
          fontsize=15, fontweight='bold', pad=25)

plt.ylim(50, 115)
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.legend(loc='lower left', frameon=True, shadow=True, fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:

# --- 1. DATA PREP (Including 90+) ---
df_sens = df_analysis.copy()
df_sens["AGE"] = pd.to_numeric(df_sens["AGE"], errors="coerce")
df_sens = df_sens[(df_sens["AGE"] >= 18) & (df_sens["AGE"] <= 105)].copy()

if df_sens["SEX"].dtype == "object":
    df_sens["SEX"] = df_sens["SEX"].map({"M": 0, "F": 1, "m": 0, "f": 1})

# Define Bins (including 85+)
bins = [18, 25, 35, 45, 55, 65, 75, 85, 106]
labels = ["18-24", "25-34", "35-44", "45-54", "55-64", "65-74", "75-84", "85+"]
df_sens["Age_Group"] = pd.cut(df_sens["AGE"], bins=bins, labels=labels, right=False)

df_reg = df_sens.dropna(subset=["Max_Effect_Concentration",  "ASA_SCORE", "BMI", "SEX"]).copy()

# Physiological Target Formula
def ce50_eleveld(age):
    return 3.08 * np.exp(-0.00635 * (age - 35))

# --- 2. MODELING ---
AGE_LO, AGE_HI = float(df_reg["AGE"].min()), float(df_reg["AGE"].max())
g_asa, g_bmi, g_sex = df_reg[["ASA_SCORE", "BMI", "SEX"]].mean()

# Models
model_ce = smf.ols(f"Max_Effect_Concentration ~ bs(AGE, df=5, lower_bound={AGE_LO}, upper_bound={AGE_HI}) + ASA_SCORE + BMI + SEX", data=df_reg).fit()
model_dose = smf.ols(f"DOSE_PER_ABW ~ bs(AGE, df=5, lower_bound={AGE_LO}, upper_bound={AGE_HI}) + ASA_SCORE + BMI + SEX", data=df_reg).fit()

# Predictions for Line Trends
age_grid = np.linspace(18, AGE_HI, 400)
grid_df = pd.DataFrame({"AGE": age_grid, "ASA_SCORE": g_asa, "BMI": g_bmi, "SEX": g_sex})
pred_grid_ce = model_ce.get_prediction(grid_df).summary_frame()

# Predictions for Binned Points
bin_stats = df_reg.groupby("Age_Group", observed=True).agg(mean_age=("AGE", "mean")).reset_index()
bin_pred_input = bin_stats.rename(columns={"mean_age": "AGE"}).assign(ASA_SCORE=g_asa, BMI=g_bmi, SEX=g_sex)
bin_pred_ce = model_ce.get_prediction(bin_pred_input).summary_frame()
bin_pred_dose = model_dose.get_prediction(bin_pred_input).summary_frame()

# --- 3. UNIFIED VISUALIZATION (Master Figure Style) ---
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 19))
plt.subplots_adjust(hspace=0.32, top=0.95, bottom=0.08, left=0.1, right=0.95)
x_axis = np.arange(len(labels))

# PANEL A: ABSOLUTE EXPOSURE
ax1.plot(age_grid, pred_grid_ce["mean"], linewidth=4, color='#1f77b4', label="Adj. Predicted Peak $C_e$")
ax1.plot(age_grid, ce50_eleveld(age_grid), linestyle="--", linewidth=3, color='red', label="Eleveld $Ce_{50}$ Target")
ax1.bar(bin_stats["mean_age"], bin_pred_ce["mean"], width=4.0, color='#5dade2', alpha=0.6, edgecolor='#1f77b4',
        yerr=[bin_pred_ce["mean"] - bin_pred_ce["mean_ci_lower"], bin_pred_ce["mean_ci_upper"] - bin_pred_ce["mean"]], capsize=5)

for i, row in bin_pred_ce.iterrows():
    target = ce50_eleveld(bin_stats["mean_age"].iloc[i])
    diff = ((row["mean"] / target) - 1) * 100
    ax1.text(bin_stats["mean_age"].iloc[i], row["mean"] + 0.15, f'{row["mean"]:.2f}', ha='center', fontweight='bold', color='#1f77b4')
    ax1.text(bin_stats["mean_age"].iloc[i], row["mean"] + 0.45, f'+{diff:.0f}%', ha='center', fontweight='bold', color='red')

ax1.set_title("A: Standardized Peak Exposure ($C_e$) vs. Validated Target", loc='left', fontsize=14, fontweight='bold')
ax1.set_ylabel("Effect-Site Concentration (mcg/mL)")
ax1.set_ylim(0, 5.5)
ax1.set_xticks(bin_stats["mean_age"])
ax1.set_xticklabels(labels)
ax1.legend(loc="upper right", frameon=True)

# PANEL B: CLINICIAN DOSE TITRATION
ax2.bar(bin_stats["mean_age"], bin_pred_dose["mean"], width=4.0, color='#a1d99b', alpha=0.6, edgecolor='#2ca02c',
        yerr=[bin_pred_dose["mean"] - bin_pred_dose["mean_ci_lower"], bin_pred_dose["mean_ci_upper"] - bin_pred_dose["mean"]], capsize=5)

for i, row in bin_pred_dose.iterrows():
    ax2.text(bin_stats["mean_age"].iloc[i], row["mean"] + 0.08, f'{row["mean"]:.2f}', ha='center', fontweight='bold', color='#2ca02c')

ax2.set_title("B: Standardized Clinician Induction Dose Titration", loc='left', fontsize=14, fontweight='bold')
ax2.set_ylabel("Induction Dose (mg/kg ABW)")
ax2.set_xticks(bin_stats["mean_age"])
ax2.set_xticklabels(labels)

# PANEL C: RELATIVE DIVERGENCE (Identical Legend Labels)
b_ce, b_dose, b_target = bin_pred_ce['mean'].iloc[0], bin_pred_dose['mean'].iloc[0], ce50_eleveld(bin_stats["mean_age"].iloc[0])

ax3.plot(x_axis, (bin_pred_dose['mean'] / b_dose) * 100, marker='o', linewidth=4, color='#2ca02c',
         label="Clinician Titration (mg/kg % Baseline)")
ax3.plot(x_axis, (bin_pred_ce['mean'] / b_ce) * 100, marker='s', linewidth=4, color='#1f77b4',
         label="Modeled Peak Exposure (Ce % Baseline)")
ax3.plot(x_axis, (ce50_eleveld(bin_stats["mean_age"]) / b_target) * 100, marker='D', linewidth=3, linestyle='--', color='red',
         label="Validated PD Reference (Ce50 % Baseline)")
ax3.fill_between(x_axis, (bin_pred_ce['mean_ci_lower']/b_ce)*100, (bin_pred_ce['mean_ci_upper']/b_ce)*100, color='#1f77b4', alpha=0.1)

ax3.set_title("C: Relative Divergence from Requirement", loc='left', fontsize=14, fontweight='bold')
ax3.set_ylabel("Percentage of Young Adult Baseline (%)")
ax3.set_xlabel("Age Cohort (Years)", fontsize=13, labelpad=10)
ax3.set_xticks(x_axis)
ax3.set_xticklabels(labels)
ax3.set_ylim(45, 115)
ax3.legend(loc='lower left', frameon=True)
ax3.grid(axis='y', linestyle='--', alpha=0.3)

# SAVE FIGURE
plt.savefig("figure_sensitivity_90plus.pdf", bbox_inches='tight', dpi=300)
plt.show()

# --- 4. DATA TABLE GENERATION (Updated with Induction Dose) ---
targets_val = ce50_eleveld(bin_stats['mean_age'])
ov_mean = ((bin_pred_ce['mean'] / targets_val) - 1) * 100
ov_low = ((bin_pred_ce['mean_ci_lower'] / targets_val) - 1) * 100
ov_high = ((bin_pred_ce['mean_ci_upper'] / targets_val) - 1) * 100

table_90plus = pd.DataFrame({
    'Age Group': labels,
    'N': [f"{int(x):,}" for x in df_reg.groupby("Age_Group", observed=True).size().values],

    # NEW COLUMN: Accurate mg/kg for this specific dataset
    'Induction Dose': [f"{m:.2f} ({l:.2f}--{u:.2f})" for m, l, u in zip(bin_pred_dose['mean'], bin_pred_dose['mean_ci_lower'], bin_pred_dose['mean_ci_upper'])],

    'Peak Ce (95% CI)': [f"{m:.2f} ({l:.2f}--{u:.2f})" for m, l, u in zip(bin_pred_ce['mean'], bin_pred_ce['mean_ci_lower'], bin_pred_ce['mean_ci_upper'])],
    'Target Ce50': [f"{t:.2f}" for t in targets_val],
    'Surplus (95% CI)': [f"{m:.1f}% ({l:.1f}--{u:.1f}%)" for m, l, u in zip(ov_mean, ov_low, ov_high)]
})

# Save to CSV (this is crucial for your Master Merge script later)
table_90plus.to_csv("table_sensitivity_90plus.csv", index=False)

# Export to LaTeX (Note: added an extra 'c' to column_format for the new column)
with open("table_sensitivity_90plus.tex", "w") as f:
    f.write(table_90plus.to_latex(index=False, escape=False, column_format='lccccc').replace('%', r'\%'))

print("✅ Success: figure_sensitivity_90plus.pdf and table_sensitivity_90plus.tex (with Dose) saved.")

In [ ]:

# --- 1. DATA PREP (Including 90+) ---
df_sens = df_analysis.copy()
df_sens["AGE"] = pd.to_numeric(df_sens["AGE"], errors="coerce")
df_sens = df_sens[(df_sens["AGE"] >= 18) & (df_sens["AGE"] <= 105)].copy()

if df_sens["SEX"].dtype == "object":
    df_sens["SEX"] = df_sens["SEX"].map({"M": 0, "F": 1, "m": 0, "f": 1})

# Define Bins (including 85+)
bins = [18, 25, 35, 45, 55, 65, 75, 85, 106]
labels = ["18-24", "25-34", "35-44", "45-54", "55-64", "65-74", "75-84", "85+"]
df_sens["Age_Group"] = pd.cut(df_sens["AGE"], bins=bins, labels=labels, right=False)

df_reg = df_sens.dropna(subset=["Max_Effect_Concentration", "DOSE_PER_ABW", "ASA_SCORE", "BMI", "SEX"]).copy()

# Physiological Target Formula
def ce50_eleveld(age):
    return 3.08 * np.exp(-0.00635 * (age - 35))

# --- 2. MODELING ---
AGE_LO, AGE_HI = float(df_reg["AGE"].min()), float(df_reg["AGE"].max())
g_asa, g_bmi, g_sex = df_reg[["ASA_SCORE", "BMI", "SEX"]].mean()

# Models
model_ce = smf.ols(f"Max_Effect_Concentration ~ bs(AGE, df=5, lower_bound={AGE_LO}, upper_bound={AGE_HI}) + ASA_SCORE + BMI + SEX", data=df_reg).fit()
model_dose = smf.ols(f"DOSE_PER_ABW ~ bs(AGE, df=5, lower_bound={AGE_LO}, upper_bound={AGE_HI}) + ASA_SCORE + BMI + SEX", data=df_reg).fit()

# Predictions for Line Trends
age_grid = np.linspace(18, AGE_HI, 400)
grid_df = pd.DataFrame({"AGE": age_grid, "ASA_SCORE": g_asa, "BMI": g_bmi, "SEX": g_sex})
pred_grid_ce = model_ce.get_prediction(grid_df).summary_frame()

# Predictions for Binned Points
bin_stats = df_reg.groupby("Age_Group", observed=True).agg(mean_age=("AGE", "mean")).reset_index()
bin_pred_input = bin_stats.rename(columns={"mean_age": "AGE"}).assign(ASA_SCORE=g_asa, BMI=g_bmi, SEX=g_sex)
bin_pred_ce = model_ce.get_prediction(bin_pred_input).summary_frame()
bin_pred_dose = model_dose.get_prediction(bin_pred_input).summary_frame()

# --- 3. UNIFIED VISUALIZATION (Master Figure Style) ---
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 19))
plt.subplots_adjust(hspace=0.32, top=0.95, bottom=0.08, left=0.1, right=0.95)
x_axis = np.arange(len(labels))

# PANEL A: ABSOLUTE EXPOSURE
ax1.plot(age_grid, pred_grid_ce["mean"], linewidth=4, color='#1f77b4', label="Adj. Predicted Peak $C_e$")
ax1.plot(age_grid, ce50_eleveld(age_grid), linestyle="--", linewidth=3, color='red', label="Eleveld $Ce_{50}$ Target")
ax1.bar(bin_stats["mean_age"], bin_pred_ce["mean"], width=4.0, color='#5dade2', alpha=0.6, edgecolor='#1f77b4',
        yerr=[bin_pred_ce["mean"] - bin_pred_ce["mean_ci_lower"], bin_pred_ce["mean_ci_upper"] - bin_pred_ce["mean"]], capsize=5)

for i, row in bin_pred_ce.iterrows():
    target = ce50_eleveld(bin_stats["mean_age"].iloc[i])
    diff = ((row["mean"] / target) - 1) * 100
    ax1.text(bin_stats["mean_age"].iloc[i], row["mean"] + 0.15, f'{row["mean"]:.2f}', ha='center', fontweight='bold', color='#1f77b4')
    ax1.text(bin_stats["mean_age"].iloc[i], row["mean"] + 0.45, f'+{diff:.0f}%', ha='center', fontweight='bold', color='red')

ax1.set_title("A: Standardized Peak Exposure ($C_e$) vs. Validated Target", loc='left', fontsize=14, fontweight='bold')
ax1.set_ylabel("Effect-Site Concentration (mcg/mL)")
ax1.set_ylim(0, 5.5)
ax1.set_xticks(bin_stats["mean_age"])
ax1.set_xticklabels(labels)
ax1.legend(loc="upper right", frameon=True)

# PANEL B: CLINICIAN DOSE TITRATION
ax2.bar(bin_stats["mean_age"], bin_pred_dose["mean"], width=4.0, color='#a1d99b', alpha=0.6, edgecolor='#2ca02c',
        yerr=[bin_pred_dose["mean"] - bin_pred_dose["mean_ci_lower"], bin_pred_dose["mean_ci_upper"] - bin_pred_dose["mean"]], capsize=5)

for i, row in bin_pred_dose.iterrows():
    ax2.text(bin_stats["mean_age"].iloc[i], row["mean"] + 0.08, f'{row["mean"]:.2f}', ha='center', fontweight='bold', color='#2ca02c')

ax2.set_title("B: Standardized Clinician Induction Dose Titration", loc='left', fontsize=14, fontweight='bold')
ax2.set_ylabel("Induction Dose (mg/kg ABW)")
ax2.set_xticks(bin_stats["mean_age"])
ax2.set_xticklabels(labels)

# PANEL C: RELATIVE DIVERGENCE (Identical Legend Labels)
b_ce, b_dose, b_target = bin_pred_ce['mean'].iloc[0], bin_pred_dose['mean'].iloc[0], ce50_eleveld(bin_stats["mean_age"].iloc[0])

ax3.plot(x_axis, (bin_pred_dose['mean'] / b_dose) * 100, marker='o', linewidth=4, color='#2ca02c',
         label="Clinician Titration (mg/kg % Baseline)")
ax3.plot(x_axis, (bin_pred_ce['mean'] / b_ce) * 100, marker='s', linewidth=4, color='#1f77b4',
         label="Modeled Peak Exposure (Ce % Baseline)")
ax3.plot(x_axis, (ce50_eleveld(bin_stats["mean_age"]) / b_target) * 100, marker='D', linewidth=3, linestyle='--', color='red',
         label="Validated PD Reference (Ce50 % Baseline)")
ax3.fill_between(x_axis, (bin_pred_ce['mean_ci_lower']/b_ce)*100, (bin_pred_ce['mean_ci_upper']/b_ce)*100, color='#1f77b4', alpha=0.1)

ax3.set_title("C: Relative Divergence from Requirement", loc='left', fontsize=14, fontweight='bold')
ax3.set_ylabel("Percentage of Young Adult Baseline (%)")
ax3.set_xlabel("Age Cohort (Years)", fontsize=13, labelpad=10)
ax3.set_xticks(x_axis)
ax3.set_xticklabels(labels)
ax3.set_ylim(45, 115)
ax3.legend(loc='lower left', frameon=True)
ax3.grid(axis='y', linestyle='--', alpha=0.3)

# SAVE FIGURE
plt.savefig("figure_sensitivity_90plus.pdf", bbox_inches='tight', dpi=300)
plt.show()

# --- 4. DATA TABLE GENERATION (Updated with Induction Dose) ---
targets_val = ce50_eleveld(bin_stats['mean_age'])
ov_mean = ((bin_pred_ce['mean'] / targets_val) - 1) * 100
ov_low = ((bin_pred_ce['mean_ci_lower'] / targets_val) - 1) * 100
ov_high = ((bin_pred_ce['mean_ci_upper'] / targets_val) - 1) * 100

table_90plus = pd.DataFrame({
    'Age Group': labels,
    'N': [f"{int(x):,}" for x in df_reg.groupby("Age_Group", observed=True).size().values],

    # NEW COLUMN: Accurate mg/kg for this specific dataset
    'Induction Dose': [f"{m:.2f} ({l:.2f}--{u:.2f})" for m, l, u in zip(bin_pred_dose['mean'], bin_pred_dose['mean_ci_lower'], bin_pred_dose['mean_ci_upper'])],

    'Peak Ce (95% CI)': [f"{m:.2f} ({l:.2f}--{u:.2f})" for m, l, u in zip(bin_pred_ce['mean'], bin_pred_ce['mean_ci_lower'], bin_pred_ce['mean_ci_upper'])],
    'Target Ce50': [f"{t:.2f}" for t in targets_val],
    'Surplus (95% CI)': [f"{m:.1f}% ({l:.1f}--{u:.1f}%)" for m, l, u in zip(ov_mean, ov_low, ov_high)]
})

# Save to CSV (this is crucial for your Master Merge script later)
table_90plus.to_csv("table_sensitivity_90plus.csv", index=False)

# Export to LaTeX (Note: added an extra 'c' to column_format for the new column)
with open("table_sensitivity_90plus.tex", "w") as f:
    f.write(table_90plus.to_latex(index=False, escape=False, column_format='lccccc').replace('%', r'\%'))

print("✅ Success: figure_sensitivity_90plus.pdf and table_sensitivity_90plus.tex (with Dose) saved.")

## Sensitivity analysis 3: Unadjusted regression models

In [ ]:
# --- 3. LOAD THE DATA ---
file_path = "INSERT_INPUT_FILE_HERE.pkl"  # e.g., primary Eleveld, Schnider, relaxed-input, or medication-adjusted dataset

if os.path.exists(file_path):
    df_analysis = pd.read_pickle(file_path)
    print(f"Loaded {len(df_analysis):,} rows from {file_path}.")
else:
    raise FileNotFoundError(
        f"Input file not found: {file_path}. "
        "Update file_path to the appropriate patient-level simulation dataset before running this section."
    )

In [ ]:

# 1. SETUP
df_raw = df_analysis.copy()
df_raw["AGE"] = pd.to_numeric(df_raw["AGE"], errors="coerce")
df_raw = df_raw[(df_raw["AGE"] >= 18) & (df_raw["AGE"] < 90)].copy()

# Synchronized Bins and Labels
bins = [18, 25, 35, 45, 55, 65, 75, 85, 90]
labels = ["18-24", "25-34", "35-44", "45-54", "55-64", "65-74", "75-84", "85-89"]
df_raw["Age_Group"] = pd.cut(df_raw["AGE"], bins=bins, labels=labels, right=False)

# 2. ARITHMETIC AGGREGATION (The "True" Raw Way)
raw_summary = df_raw.groupby('Age_Group', observed=True).agg({
    'Max_Effect_Concentration': ['mean', 'sem'],
    'DOSE_PER_ABW': ['mean', 'sem'],
    'AGE': 'mean'
}).reset_index()

# Flatten columns for easy plotting
raw_summary.columns = ['Age_Group', 'Ce_M', 'Ce_SE', 'Dose_M', 'Dose_SE', 'Age_Mean']

# Calculate 95% Confidence Intervals (Arithmetic)
raw_summary['Ce_L'] = raw_summary['Ce_M'] - (1.96 * raw_summary['Ce_SE'])
raw_summary['Ce_U'] = raw_summary['Ce_M'] + (1.96 * raw_summary['Ce_SE'])
raw_summary['Dose_L'] = raw_summary['Dose_M'] - (1.96 * raw_summary['Dose_SE'])
raw_summary['Dose_U'] = raw_summary['Dose_M'] + (1.96 * raw_summary['Dose_SE'])

# Calculate Targets based on the mean age of each bin
def ce50_eleveld(age):
    return 3.08 * np.exp(-0.00635 * (age - 35))

raw_summary['Target'] = ce50_eleveld(raw_summary['Age_Mean'])

In [ ]:
plt.figure(figsize=(15, 8.5))
x_axis = np.arange(len(labels))

# Bars based on arithmetic means
plt.bar(x_axis, raw_summary['Ce_M'], yerr=[raw_summary['Ce_M']-raw_summary['Ce_L'], raw_summary['Ce_U']-raw_summary['Ce_M']],
        width=0.6, color='#5dade2', alpha=0.6, edgecolor='#1f77b4', capsize=5, label='Observed Peak $C_e$ (Arithmetic Mean)')

# Red Target Line
plt.plot(x_axis, raw_summary['Target'], marker='D', color='red', linestyle='--', linewidth=3, label='Eleveld $Ce_{50}$ Target')

# Value Labels and Surplus Calculation
for i in range(len(raw_summary)):
    val = raw_summary['Ce_M'].iloc[i]
    targ = raw_summary['Target'].iloc[i]
    diff = ((val / targ) - 1) * 100
    plt.text(i, val + 0.1, f'{val:.2f}', ha='center', fontweight='bold', color='#1f77b4')
    plt.text(i, val + 0.35, f'+{diff:.0f}%', ha='center', fontweight='bold', color='red')

plt.xticks(x_axis, labels)
plt.title("Figure A: RAW Peak Brain Exposure ($C_e$) vs. Validated Target\n(Arithmetic Means - No Mathematical Smoothing)", fontsize=15, fontweight='bold', pad=40)
plt.ylabel("Effect-Site Concentration (mcg/mL)")
plt.ylim(0, 5)
plt.legend(loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(15, 8.5))
x_axis = np.arange(len(labels))

# Bars based on arithmetic means
plt.bar(x_axis, raw_summary['Dose_M'], yerr=[raw_summary['Dose_M']-raw_summary['Dose_L'], raw_summary['Dose_U']-raw_summary['Dose_M']],
        width=0.6, color='#a1d99b', alpha=0.6, edgecolor='#2ca02c', capsize=5, label='Observed Induction Dose')

# Value Labels
for i in range(len(raw_summary)):
    val = raw_summary['Dose_M'].iloc[i]
    plt.text(i, val + 0.08, f'{val:.2f}', ha='center', fontweight='bold', color='#2ca02c')

plt.xticks(x_axis, labels)
plt.title("Figure B: RAW Clinician Induction Dose Titration\n(Arithmetic Means - Unadjusted)", fontsize=15, fontweight='bold', pad=30)
plt.ylabel("Induction Dose (mg/kg ABW)")
plt.ylim(0, raw_summary['Dose_U'].max() + 0.5)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 8))
x_axis = np.arange(len(labels))

# Normalized to 18-24 Baseline
b_ce, b_dose, b_target = raw_summary['Ce_M'].iloc[0], raw_summary['Dose_M'].iloc[0], raw_summary['Target'].iloc[0]

# Line 1: Dose Decay
plt.plot(x_axis, (raw_summary['Dose_M']/b_dose)*100, marker='o', linewidth=4, color='#2ca02c', label="Raw Dose (% Baseline)")

# Line 2: Exposure Stability
plt.plot(x_axis, (raw_summary['Ce_M']/b_ce)*100, marker='s', linewidth=4, color='#1f77b4', label="Raw Exposure ($C_e$ % Baseline)")

# Line 3: Target Requirement
plt.plot(x_axis, (raw_summary['Target']/b_target)*100, marker='D', linestyle='--', linewidth=3, color='red', label="Physiological Target Line")

plt.xticks(x_axis, labels)
plt.ylabel("Percentage of Young Adult (18-24) Baseline (%)", fontsize=12)
plt.title("Figure C: RAW Relative Divergence\n(Arithmetic Normalization - Clinical Titration vs. Requirement)", fontsize=14, fontweight='bold', pad=20)
plt.ylim(50, 115)
plt.legend(loc='lower left', frameon=True)
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# --- 1. DATA PREP: Create df_raw and Age_Group ---
df_raw = df_analysis.copy()
df_raw["AGE"] = pd.to_numeric(df_raw["AGE"], errors="coerce")
df_raw = df_raw[(df_raw["AGE"] >= 18) & (df_raw["AGE"] < 90)].copy()

# Define Synchronized Bins and Labels
bins = [18, 25, 35, 45, 55, 65, 75, 85, 90]
labels = ["18-24", "25-34", "35-44", "45-54", "55-64", "65-74", "75-84", "85-89"]
df_raw["Age_Group"] = pd.cut(df_raw["AGE"], bins=bins, labels=labels, right=False)

# --- 2. ARITHMETIC AGGREGATION ---
# Group by df_raw (which now has 'Age_Group')
raw_stats = df_raw.groupby('Age_Group', observed=True).agg({
    'Max_Effect_Concentration': ['mean', 'sem', 'size'],
    'DOSE_PER_ABW': ['mean', 'sem'],
    'AGE': 'mean'
}).reset_index()

# Flatten columns
raw_stats.columns = ['Age_Group', 'Ce_M', 'Ce_SE', 'N', 'Dose_M', 'Dose_SE', 'Age_Mean']

# Calculate 95% Confidence Intervals
raw_stats['Ce_L'] = raw_stats['Ce_M'] - (1.96 * raw_stats['Ce_SE'])
raw_stats['Ce_U'] = raw_stats['Ce_M'] + (1.96 * raw_stats['Ce_SE'])
raw_stats['Dose_L'] = raw_stats['Dose_M'] - (1.96 * raw_stats['Dose_SE'])
raw_stats['Dose_U'] = raw_stats['Dose_M'] + (1.96 * raw_stats['Dose_SE'])

# Calculate Target Requirement
def ce50_eleveld(age):
    return 3.08 * np.exp(-0.00635 * (age - 35))

raw_stats['Target'] = raw_stats['Age_Mean'].apply(ce50_eleveld)
raw_summary = raw_stats.copy()

# --- 3. CALCULATIONS FOR PANEL C (Consistent with Master Figure) ---
b_ce = raw_summary['Ce_M'].iloc[0]
b_dose = raw_summary['Dose_M'].iloc[0]
b_target = raw_summary['Target'].iloc[0]

# Normalized Percentages
dose_pct = (raw_summary['Dose_M'] / b_dose) * 100
ce_pct = (raw_summary['Ce_M'] / b_ce) * 100
target_pct = (raw_summary['Target'] / b_target) * 100
ce_l_pct = (raw_summary['Ce_L'] / b_ce) * 100
ce_u_pct = (raw_summary['Ce_U'] / b_ce) * 100

# --- 4. VISUALIZATION (Height=19, hspace=0.32) ---
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 19))
plt.subplots_adjust(hspace=0.32, top=0.95, bottom=0.08, left=0.1, right=0.95)
x_axis = np.arange(len(labels))

# PANEL A: RAW EXPOSURE
ax1.bar(x_axis, raw_summary['Ce_M'],
        yerr=[raw_summary['Ce_M']-raw_summary['Ce_L'], raw_summary['Ce_U']-raw_summary['Ce_M']],
        width=0.6, color='#5dade2', alpha=0.6, edgecolor='#1f77b4', capsize=5,
        label='Observed Peak $C_e$')
ax1.plot(x_axis, raw_summary['Target'], marker='D', color='red', linestyle='--', linewidth=3, label='Eleveld $Ce_{50}$ Target')

for i in range(len(raw_summary)):
    val = raw_summary['Ce_M'].iloc[i]
    targ = raw_summary['Target'].iloc[i]
    diff = ((val / targ) - 1) * 100
    ax1.text(i, val + 0.1, f'{val:.2f}', ha='center', fontweight='bold', color='#1f77b4')
    ax1.text(i, val + 0.38, f'+{diff:.0f}%', ha='center', fontweight='bold', color='red')

ax1.set_title("A: RAW Peak Exposure ($C_e$) vs. Validated Target", loc='left', fontsize=14, fontweight='bold')
ax1.set_ylabel("Effect-Site Concentration (mcg/mL)")
ax1.set_ylim(0, 5.5)
ax1.set_xticks(x_axis)
ax1.set_xticklabels(labels)
ax1.legend(loc="upper right", frameon=True)

# PANEL B: RAW TITRATION
ax2.bar(x_axis, raw_summary['Dose_M'],
        yerr=[raw_summary['Dose_M']-raw_summary['Dose_L'], raw_summary['Dose_U']-raw_summary['Dose_M']],
        width=0.6, color='#a1d99b', alpha=0.6, edgecolor='#2ca02c', capsize=5,
        label='Observed Induction Dose')

for i in range(len(raw_summary)):
    val = raw_summary['Dose_M'].iloc[i]
    ax2.text(i, val + 0.08, f'{val:.2f}', ha='center', fontweight='bold', color='#2ca02c')

ax2.set_title("B: RAW Clinician Induction Dose Titration (Unadjusted)", loc='left', fontsize=14, fontweight='bold')
ax2.set_ylabel("Induction Dose (mg/kg ABW)")
ax2.set_ylim(0, raw_summary['Dose_U'].max() + 0.5)
ax2.set_xticks(x_axis)
ax2.set_xticklabels(labels)

# PANEL C: RAW RELATIVE DIVERGENCE (Master Legend Labels)
ax3.plot(x_axis, dose_pct, marker='o', linewidth=4, color='#2ca02c',
         label="Clinician Titration (mg/kg % Baseline)")
ax3.plot(x_axis, ce_pct, marker='s', linewidth=4, color='#1f77b4',
         label="Modeled Peak Exposure (Ce % Baseline)")
ax3.plot(x_axis, target_pct, marker='D', linestyle='--', linewidth=3, color='red',
         label="Validated PD Reference (Ce50 % Baseline)")
ax3.fill_between(x_axis, ce_l_pct, ce_u_pct, color='#1f77b4', alpha=0.1)

ax3.set_title("C: RAW Relative Divergence (Arithmetic Normalization)", loc='left', fontsize=14, fontweight='bold')
ax3.set_ylabel("Percentage of Baseline (%)")
ax3.set_xlabel("Age Cohort (Years)", fontsize=13, labelpad=10)
ax3.set_xticks(x_axis)
ax3.set_xticklabels(labels)
ax3.set_ylim(40, 115)
ax3.legend(loc='lower left', frameon=True)
ax3.grid(axis='y', linestyle='--', alpha=0.3)

# SAVE FIGURE
plt.savefig("figure_sensitivity_raw.pdf", bbox_inches='tight', dpi=300)
plt.show()

# --- 5. DATA TABLE GENERATION (Updated with Induction Dose) ---
ov_mean = ((raw_summary['Ce_M'] / raw_summary['Target']) - 1) * 100
ov_low = ((raw_summary['Ce_L'] / raw_summary['Target']) - 1) * 100
ov_high = ((raw_summary['Ce_U'] / raw_summary['Target']) - 1) * 100

table_raw = pd.DataFrame({
    'Age Group': labels,
    'N': [f"{int(x):,}" for x in raw_summary['N']],

    # NEW COLUMN: Raw Arithmetic Induction Dose (mg/kg)
    'Induction Dose': [f"{m:.2f} ({l:.2f}--{u:.2f})" for m, l, u in zip(raw_summary['Dose_M'], raw_summary['Dose_L'], raw_summary['Dose_U'])],

    'Peak Ce (95% CI)': [f"{m:.2f} ({l:.2f}--{u:.2f})" for m, l, u in zip(raw_summary['Ce_M'], raw_summary['Ce_L'], raw_summary['Ce_U'])],
    'Target Ce50': [f"{t:.2f}" for t in raw_summary['Target']],
    'Surplus (95% CI)': [f"{m:.1f}% ({l:.1f}--{u:.1f}%)" for m, l, u in zip(ov_mean, ov_low, ov_high)]
})

# Save to CSV for the Master Merge
table_raw.to_csv("table_sensitivity_raw.csv", index=False)

# Export to LaTeX
with open("table_sensitivity_raw.tex", "w") as f:
    # Changed column_format to lccccc for 6 columns
    f.write(table_raw.to_latex(index=False, escape=False, column_format='lccccc').replace('%', r'\%'))

print("✅ Success: 'figure_sensitivity_raw.pdf' and 'table_sensitivity_raw.tex' (with Dose) saved.")

## Sensitivity analysis 4: Broader-input cohort with relaxed preprocessing

In [ ]:
# --- 3. LOAD THE DATA ---
file_path = "INSERT_INPUT_FILE_HERE.pkl"  # e.g., primary Eleveld, Schnider, relaxed-input, or medication-adjusted dataset

if os.path.exists(file_path):
    df_analysis = pd.read_pickle(file_path)
    print(f"Loaded {len(df_analysis):,} rows from {file_path}.")
else:
    raise FileNotFoundError(
        f"Input file not found: {file_path}. "
        "Update file_path to the appropriate patient-level simulation dataset before running this section."
    )

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# 1. PREPARATION (18-89 Only with Attrition Reporting)
# ---------------------------------------------------
df = df_analysis.copy()

# Ensure AGE is numeric and capture initial unique patient count
df["AGE"] = pd.to_numeric(df["AGE"], errors="coerce")
initial_patients = df['OR_CASE_ID'].nunique()

# Step A: Filter for 18+
df = df[df["AGE"] >= 18].copy()
under_18_removed = initial_patients - df['OR_CASE_ID'].nunique()
post_pediatric_count = df['OR_CASE_ID'].nunique()

# Step B: Filter for < 90 (The specific request)
df = df[df["AGE"] < 90].copy()
over_90_removed = post_pediatric_count - df['OR_CASE_ID'].nunique()

# Print the attrition results for your STROBE diagram/Methods section
print(f"--- Attrition Report ---")
print(f"Initial unique patients: {initial_patients:,}")
print(f"Removed pediatric cases (<18): {under_18_removed:,}")
print(f"Removed geriatric outliers (>=90): {over_90_removed:,}")
print(f"Final analytic cohort (18-89): {df['OR_CASE_ID'].nunique():,}")
print("-" * 25)

# Standardize Sex (0=Male, 1=Female)
if df["SEX"].dtype == "object":
    df["SEX"] = df["SEX"].map({"M": 0, "F": 1, "m": 0, "f": 1})

# Define Bins (Note: upper bound 90 ensures 85-89 group is captured)
bins = [18, 25, 35, 45, 55, 65, 75, 85, 90]
labels = ["18-24", "25-34", "35-44", "45-54", "55-64", "65-74", "75-84", "85-89"]
df["Age_Group"] = pd.cut(df["AGE"], bins=bins, labels=labels, right=False)

AGE_LO, AGE_HI = float(df["AGE"].min()), float(df["AGE"].max())

In [ ]:

# Eleveld Target Formula
def ce50_eleveld(age):
    return 3.08 * np.exp(-0.00635 * (age - 35))

# 1) PREPARATION (18-89 Only)
# ----------------------------
df = df_analysis.copy()
df["AGE"] = pd.to_numeric(df["AGE"], errors="coerce")
df = df[(df["AGE"] >= 18) & (df["AGE"] < 90)].copy()

if df["SEX"].dtype == "object":
    df["SEX"] = df["SEX"].map({"M": 0, "F": 1, "m": 0, "f": 1})

bins   = [18, 25, 35, 45, 55, 65, 75, 85, 90]
labels = ["18-24", "25-34", "35-44", "45-54", "55-64", "65-74", "75-84", "85-89"]
df["Age_Group"] = pd.cut(df["AGE"], bins=bins, labels=labels, right=False)

df_reg = df.dropna(subset=["Max_Effect_Concentration", "AGE", "ASA_SCORE", "BMI", "SEX"]).copy()

# 2) MODELING
# ------------
AGE_LO, AGE_HI = float(df_reg["AGE"].min()), float(df_reg["AGE"].max())
formula_ce = f"Max_Effect_Concentration ~ bs(AGE, df=5, lower_bound={AGE_LO}, upper_bound={AGE_HI}) + ASA_SCORE + BMI + SEX"
model_ce = smf.ols(formula_ce, data=df_reg).fit()

g_asa, g_bmi, g_sex = df_reg[["ASA_SCORE", "BMI", "SEX"]].mean()

# Predictions
age_grid = np.linspace(18, 89, 400)
grid_df = pd.DataFrame({"AGE": age_grid, "ASA_SCORE": g_asa, "BMI": g_bmi, "SEX": g_sex})
pred_grid = model_ce.get_prediction(grid_df).summary_frame()

bin_stats = df_reg.groupby("Age_Group", observed=True).agg(mean_age=("AGE", "mean")).reset_index()
bin_pred_ce = model_ce.get_prediction(bin_stats.rename(columns={"mean_age": "AGE"}).assign(ASA_SCORE=g_asa, BMI=g_bmi, SEX=g_sex)).summary_frame()

# 3) VISUALIZATION: EXPOSURE
# ----------------------------
plt.figure(figsize=(15, 8.5)) # Increased height slightly
plt.plot(age_grid, pred_grid["mean"], linewidth=4, color='#1f77b4', label="Adj. Predicted Peak $C_e$")
plt.plot(age_grid, ce50_eleveld(age_grid), linestyle="--", linewidth=3, color='red', label="Eleveld $Ce_{50}$ Target")

plt.bar(bin_stats["mean_age"], bin_pred_ce["mean"], width=3.5, color='#5dade2', alpha=0.6, edgecolor='#1f77b4',
        yerr=[bin_pred_ce["mean"] - bin_pred_ce["mean_ci_lower"], bin_pred_ce["mean_ci_upper"] - bin_pred_ce["mean"]], capsize=5)

for i, row in bin_pred_ce.iterrows():
    target = ce50_eleveld(bin_stats["mean_age"].iloc[i])
    diff = ((row["mean"] / target) - 1) * 100
    # Mean Value Label
    plt.text(bin_stats["mean_age"].iloc[i], row["mean"] + 0.12, f'{row["mean"]:.2f}', ha='center', fontsize=11, fontweight='bold', color='#1f77b4')
    # Surplus Percentage Label (Higher padding)
    plt.text(bin_stats["mean_age"].iloc[i], row["mean"] + 0.38, f'+{diff:.0f}%', ha='center', fontsize=11, fontweight='bold', color='red')

plt.xticks(bin_stats["mean_age"], labels)
plt.title("Standardized Peak Exposure ($C_e$) vs. Validated Target\n", fontsize=15, fontweight='bold', pad=40)
plt.ylabel("Effect-Site Concentration (mcg/mL)")
plt.ylim(0, 5.0) # Explicitly set headroom to prevent label clipping
plt.legend(bbox_to_anchor=(1.04, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# 1) MODELING DOSE
# -----------------
formula_dose = f"DOSE_PER_ABW ~ bs(AGE, df=5, lower_bound={AGE_LO}, upper_bound={AGE_HI}) + ASA_SCORE + BMI + SEX"
model_dose = smf.ols(formula_dose, data=df_reg).fit()

pred_grid_dose = model_dose.get_prediction(grid_df).summary_frame()
bin_pred_dose = model_dose.get_prediction(bin_stats.rename(columns={"mean_age": "AGE"}).assign(ASA_SCORE=g_asa, BMI=g_bmi, SEX=g_sex)).summary_frame()

# 2) VISUALIZATION: TITRATION (No Reference Lines)
# ----------------------------
plt.figure(figsize=(15, 8))
plt.plot(age_grid, pred_grid_dose["mean"], linewidth=4, color='#2ca02c', label="Adj. Induction Dose")
plt.bar(bin_stats["mean_age"], bin_pred_dose["mean"], width=3.5, color='#a1d99b', alpha=0.6, edgecolor='#2ca02c',
        yerr=[bin_pred_dose["mean"] - bin_pred_dose["mean_ci_lower"], bin_pred_dose["mean_ci_upper"] - bin_pred_dose["mean"]], capsize=5)

for i, row in bin_pred_dose.iterrows():
    plt.text(bin_stats["mean_age"].iloc[i], row["mean"] + 0.08, f'{row["mean"]:.2f}', ha='center', fontsize=11, fontweight='bold', color='#2ca02c')

plt.xticks(bin_stats["mean_age"], labels)
plt.title("Standardized Clinician Induction Dose Titration (Ages 18-89)\n", fontsize=15, fontweight='bold', pad=20)
plt.ylabel("Induction Dose (mg/kg ABW)")
plt.ylim(0, max(bin_pred_dose["mean_ci_upper"]) + 0.5) # Dynamic headroom
plt.legend(bbox_to_anchor=(1.04, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:

# 1. SETUP & STANDARDIZED MODELS
# ---------------------------------------------------------------------------
df = df_analysis.copy()
df["AGE"] = pd.to_numeric(df["AGE"], errors="coerce")
df = df[(df["AGE"] >= 18) & (df["AGE"] < 90)].copy()
if df["SEX"].dtype == "object": df["SEX"] = df["SEX"].map({"M":0,"F":1,"m":0,"f":1})

bins = [18, 25, 35, 45, 55, 65, 75, 85, 90]
labels = ["18-24", "25-34", "35-44", "45-54", "55-64", "65-74", "75-84", "85-89"]
df["Age_Group"] = pd.cut(df["AGE"], bins=bins, labels=labels, right=False)

# Fit Models
AGE_LO, AGE_HI = float(df["AGE"].min()), float(df["AGE"].max())
model_ce = smf.ols(f"Max_Effect_Concentration ~ bs(AGE, df=5, lower_bound={AGE_LO}, upper_bound={AGE_HI}) + ASA_SCORE + BMI + SEX", data=df).fit()
model_dose = smf.ols(f"DOSE_PER_ABW ~ bs(AGE, df=5, lower_bound={AGE_LO}, upper_bound={AGE_HI}) + ASA_SCORE + BMI + SEX", data=df).fit()

# 2. CALCULATE NORMALIZED % DECAY
# ---------------------------------------------------------------------------
g_means = df[["ASA_SCORE", "BMI", "SEX"]].mean()
bin_stats = df.groupby("Age_Group", observed=True).agg(mean_age=("AGE", "mean")).reset_index()
pred_df = bin_stats.rename(columns={"mean_age": "AGE"}).assign(ASA_SCORE=g_means[0], BMI=g_means[1], SEX=g_means[2])

# Predictions with CIs
ce_res = model_ce.get_prediction(pred_df).summary_frame()
dose_res = model_dose.get_prediction(pred_df).summary_frame()
target_ce50 = 3.08 * np.exp(-0.00635 * (bin_stats["mean_age"] - 35))

# Normalized to 18-24 group
b_ce, b_dose, b_target = ce_res['mean'].iloc[0], dose_res['mean'].iloc[0], target_ce50.iloc[0]

bin_stats["Dose_Pct"] = (dose_res['mean'] / b_dose) * 100
bin_stats["Ce_Pct"] = (ce_res['mean'] / b_ce) * 100
bin_stats["Ce_L"], bin_stats["Ce_U"] = (ce_res['mean_ci_lower'] / b_ce) * 100, (ce_res['mean_ci_upper'] / b_ce) * 100
bin_stats["Target_Pct"] = (target_ce50 / b_target) * 100

# 3. VISUALIZATION (PEER-REVIEW READY)
# ---------------------------------------------------------------------------
plt.figure(figsize=(14, 8))
x = np.arange(len(labels))

# Plot normalized lines
plt.plot(x, bin_stats["Dose_Pct"], marker='o', linewidth=4, color='#2ca02c', label="Clinician Titration (Induction $mg/kg$ % Baseline)")

# Blue line with Confidence Interval Band (Addressing Critique #3)
plt.plot(x, bin_stats["Ce_Pct"], marker='s', linewidth=4, color='#1f77b4', label="Modeled Peak Exposure (Predicted $C_e$ % Baseline)")
plt.fill_between(x, bin_stats["Ce_L"], bin_stats["Ce_U"], color='#1f77b4', alpha=0.15)

plt.plot(x, bin_stats["Target_Pct"], marker='D', linewidth=3, linestyle='--', color='red', label="Validated Age-Adjusted PD Reference ($Ce_{50}$ % Baseline)")

plt.xticks(x, labels)
plt.ylabel("Percentage of Young Adult (18-24) Baseline (%)", fontsize=12)
plt.title("Relative Divergence: Clinical Titration vs. Pharmacological Target\n(Standardized to Mean Covariates and 18-24 Baseline)", fontsize=14, fontweight='bold', pad=20)
plt.ylim(50, 115)
plt.legend(loc='lower left', frameon=True, facecolor='white', framealpha=0.9)
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:

# --- 1. DATA PREP (Including Outliers) ---
# We use df_analysis (the unfiltered dataset)
df_sens = df_analysis.copy()
df_sens["AGE"] = pd.to_numeric(df_sens["AGE"], errors="coerce")
df_sens = df_sens[(df_sens["AGE"] >= 18) & (df_sens["AGE"] < 90)].copy()

if df_sens["SEX"].dtype == "object":
    df_sens["SEX"] = df_sens["SEX"].map({"M": 0, "F": 1, "m": 0, "f": 1})

# Define Bins & Labels
bins = [18, 25, 35, 45, 55, 65, 75, 85, 90]
labels = ["18-24", "25-34", "35-44", "45-54", "55-64", "65-74", "75-84", "85-89"]
df_sens["Age_Group"] = pd.cut(df_sens["AGE"], bins=bins, labels=labels, right=False)

# Drop only missing covariate data
df_reg_out = df_sens.dropna(subset=["Max_Effect_Concentration", "DOSE_PER_ABW", "ASA_SCORE", "BMI", "SEX"]).copy()

# --- 2. MODELING ---
AGE_LO, AGE_HI = float(df_reg_out["AGE"].min()), float(df_reg_out["AGE"].max())
g_asa, g_bmi, g_sex = df_reg_out[["ASA_SCORE", "BMI", "SEX"]].mean()

# Models fitted on the UNFILTERED data
model_ce = smf.ols(f"Max_Effect_Concentration ~ bs(AGE, df=5, lower_bound={AGE_LO}, upper_bound={AGE_HI}) + ASA_SCORE + BMI + SEX", data=df_reg_out).fit()
model_dose = smf.ols(f"DOSE_PER_ABW ~ bs(AGE, df=5, lower_bound={AGE_LO}, upper_bound={AGE_HI}) + ASA_SCORE + BMI + SEX", data=df_reg_out).fit()

# Predictions for Line Trends
age_grid = np.linspace(18, 89, 400)
grid_df = pd.DataFrame({"AGE": age_grid, "ASA_SCORE": g_asa, "BMI": g_bmi, "SEX": g_sex})
pred_grid_ce = model_ce.get_prediction(grid_df).summary_frame()
pred_grid_dose = model_dose.get_prediction(grid_df).summary_frame()

# Predictions for Binned Points
bin_stats = df_reg_out.groupby("Age_Group", observed=True).agg(mean_age=("AGE", "mean")).reset_index()
bin_pred_input = bin_stats.rename(columns={"mean_age": "AGE"}).assign(ASA_SCORE=g_asa, BMI=g_bmi, SEX=g_sex)
bin_pred_ce = model_ce.get_prediction(bin_pred_input).summary_frame()
bin_pred_dose = model_dose.get_prediction(bin_pred_input).summary_frame()

# --- 3. CALCULATIONS FOR PANEL C ---
def ce50_eleveld(age): return 3.08 * np.exp(-0.00635 * (age - 35))

b_ce, b_dose, b_target = bin_pred_ce['mean'].iloc[0], bin_pred_dose['mean'].iloc[0], ce50_eleveld(bin_stats["mean_age"].iloc[0])

# Normalization for Panel C
bin_stats["Dose_Pct"] = (bin_pred_dose["mean"] / b_dose) * 100
bin_stats["Ce_Pct"] = (bin_pred_ce["mean"] / b_ce) * 100
bin_stats["Target_Pct"] = (ce50_eleveld(bin_stats["mean_age"]) / b_target) * 100
bin_stats["Ce_L"] = (bin_pred_ce["mean_ci_lower"] / b_ce) * 100
bin_stats["Ce_U"] = (bin_pred_ce["mean_ci_upper"] / b_ce) * 100

# --- 4. VISUALIZATION (Master Figure Style) ---
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 19))
plt.subplots_adjust(hspace=0.32, top=0.95, bottom=0.08, left=0.1, right=0.95)

# PANEL A: EXPOSURE VS REQUIREMENT
ax1.plot(age_grid, pred_grid_ce["mean"], linewidth=4, color='#1f77b4', label="Adj. Predicted Peak $C_e$")
ax1.plot(age_grid, ce50_eleveld(age_grid), linestyle="--", linewidth=3, color='red', label="Eleveld $Ce_{50}$ Target")
ax1.bar(bin_stats["mean_age"], bin_pred_ce["mean"], width=3.5, color='#5dade2', alpha=0.6, edgecolor='#1f77b4',
        yerr=[bin_pred_ce["mean"] - bin_pred_ce["mean_ci_lower"], bin_pred_ce["mean_ci_upper"] - bin_pred_ce["mean"]], capsize=5)

for i, row in bin_pred_ce.iterrows():
    target = ce50_eleveld(bin_stats["mean_age"].iloc[i])
    diff = ((row["mean"] / target) - 1) * 100
    ax1.text(bin_stats["mean_age"].iloc[i], row["mean"] + 0.12, f'{row["mean"]:.2f}', ha='center', fontweight='bold', color='#1f77b4')
    ax1.text(bin_stats["mean_age"].iloc[i], row["mean"] + 0.38, f'+{diff:.0f}%', ha='center', fontweight='bold', color='red')

ax1.set_title("A: Standardized Peak Exposure ($C_e$) vs. Validated Target", loc='left', fontsize=14, fontweight='bold')
ax1.set_ylabel("Effect-Site Concentration (mcg/mL)")
ax1.set_ylim(0, 5.5)
ax1.set_xticks(bin_stats["mean_age"])
ax1.set_xticklabels(labels)
ax1.legend(loc="upper right", frameon=True)

# PANEL B: TITRATION
ax2.plot(age_grid, pred_grid_dose["mean"], linewidth=4, color='#2ca02c', label="Adj. Induction Dose")
ax2.bar(bin_stats["mean_age"], bin_pred_dose["mean"], width=3.5, color='#a1d99b', alpha=0.6, edgecolor='#2ca02c',
        yerr=[bin_pred_dose["mean"] - bin_pred_dose["mean_ci_lower"], bin_pred_dose["mean_ci_upper"] - bin_pred_dose["mean"]], capsize=5)

for i, row in bin_pred_dose.iterrows():
    ax2.text(bin_stats["mean_age"].iloc[i], row["mean"] + 0.08, f'{row["mean"]:.2f}', ha='center', fontweight='bold', color='#2ca02c')

ax2.set_title("B: Standardized Clinician Induction Dose Titration", loc='left', fontsize=14, fontweight='bold')
ax2.set_ylabel("Induction Dose (mg/kg ABW)")
ax2.set_xticks(bin_stats["mean_age"])
ax2.set_xticklabels(labels)
ax2.legend(loc="upper right", frameon=True)

# PANEL C: RELATIVE DIVERGENCE (Master Legend Labels)
x_indices = np.arange(len(labels))
ax3.plot(x_indices, bin_stats["Dose_Pct"], marker='o', linewidth=4, color='#2ca02c',
         label="Clinician Titration (mg/kg % Baseline)")
ax3.plot(x_indices, bin_stats["Ce_Pct"], marker='s', linewidth=4, color='#1f77b4',
         label="Modeled Peak Exposure (Ce % Baseline)")
ax3.fill_between(x_indices, bin_stats["Ce_L"], bin_stats["Ce_U"], color='#1f77b4', alpha=0.15)
ax3.plot(x_indices, bin_stats["Target_Pct"], marker='D', linewidth=3, linestyle='--', color='red',
         label="Validated PD Reference (Ce50 % Baseline)")

ax3.set_title("C: Relative Divergence from Requirement", loc='left', fontsize=14, fontweight='bold')
ax3.set_ylabel("Percentage of Baseline (%)")
ax3.set_xlabel("Age Cohort (Years)", fontsize=13, labelpad=10)
ax3.set_xticks(x_indices)
ax3.set_xticklabels(labels)
ax3.set_ylim(40, 115)
ax3.legend(loc='lower left', frameon=True)
ax3.grid(axis='y', linestyle='--', alpha=0.3)

# SAVE AS PDF
plt.savefig("figure_sensitivity_outliers.pdf", bbox_inches='tight', dpi=300)
plt.show()

# --- 5. DATA TABLE GENERATION (Updated with Outlier Induction Doses) ---
targets_val = ce50_eleveld(bin_stats['mean_age'])
ov_mean = ((bin_pred_ce['mean'] / targets_val) - 1) * 100
ov_low = ((bin_pred_ce['mean_ci_lower'] / targets_val) - 1) * 100
ov_high = ((bin_pred_ce['mean_ci_upper'] / targets_val) - 1) * 100

table_out = pd.DataFrame({
    'Age Group': labels,
    'N': [f"{int(x):,}" for x in df_reg_out.groupby("Age_Group", observed=True).size().values],

    # NEW COLUMN: Induction Dose including outliers
    'Induction Dose': [f"{m:.2f} ({l:.2f}--{u:.2f})" for m, l, u in zip(bin_pred_dose['mean'], bin_pred_dose['mean_ci_lower'], bin_pred_dose['mean_ci_upper'])],

    'Peak Ce (95% CI)': [f"{m:.2f} ({l:.2f}--{u:.2f})" for m, l, u in zip(bin_pred_ce['mean'], bin_pred_ce['mean_ci_lower'], bin_pred_ce['mean_ci_upper'])],
    'Target Ce50': [f"{t:.2f}" for t in targets_val],
    'Surplus (95% CI)': [f"{m:.1f}% ({l:.1f}--{u:.1f}%)" for m, l, u in zip(ov_mean, ov_low, ov_high)]
})

# Save to CSV for the Master Merge script
table_out.to_csv("table_sensitivity_outliers.csv", index=False)

# Export to LaTeX
with open("table_sensitivity_outliers.tex", "w") as f:
    # column_format updated to 'lccccc' for 6 columns
    f.write(table_out.to_latex(index=False, escape=False, column_format='lccccc').replace('%', r'\%'))

print("✅ Success: 'figure_sensitivity_outliers.pdf' and 'table_sensitivity_outliers.tex' (with Outlier Doses) saved.")

## Sensitivity analysis 5: Adjustment for co-administered induction medications

In [ ]:
# --- 3. LOAD THE DATA ---
file_path = "INSERT_INPUT_FILE_HERE.pkl"  # e.g., primary Eleveld, Schnider, relaxed-input, or medication-adjusted dataset

if os.path.exists(file_path):
    df_analysis = pd.read_pickle(file_path)
    print(f"Loaded {len(df_analysis):,} rows from {file_path}.")
else:
    raise FileNotFoundError(
        f"Input file not found: {file_path}. "
        "Update file_path to the appropriate patient-level simulation dataset before running this section."
    )

In [ ]:

# =========================================================
# 1) DATA CLEANING
# =========================================================
df_merged = df_merged.copy()

# Convert Sex to numeric (0=Male, 1=Female)
if df_merged["SEX"].dtype == "object":
    df_merged["SEX"] = df_merged["SEX"].str.upper().map({"M": 0, "F": 1})

# Convert age to numeric
df_merged["AGE"] = pd.to_numeric(df_merged["AGE"], errors="coerce")

# Fill missing adjunct totals with 0 after left join
adjunct_cols = ["TOTAL_FENT_MCG", "TOTAL_KETA_MG", "TOTAL_MIDA_MG", "TOTAL_ETOM_MG"]
df_merged[adjunct_cols] = df_merged[adjunct_cols].fillna(0)

# Apply age filter
df_merged = df_merged[(df_merged["AGE"] >= 18) & (df_merged["AGE"] < 90)].copy()

# B) Remove clinical outliers as requested
df_merged = df_merged[
    (df_merged["TOTAL_KETA_MG"] <= 300) &
    (df_merged["TOTAL_ETOM_MG"] <= 60)  &
    (df_merged["TOTAL_FENT_MCG"] <= 1000)
].copy()

# Drop missing core analysis variables only
df_merged = df_merged.dropna(
    subset=["Max_Effect_Concentration", "AGE", "ASA_SCORE", "BMI", "SEX", "DOSE_PER_ABW"]
).copy()

print("Final regression N:", len(df_merged))
print("Final regression unique IDs:", df_merged["OR_CASE_ID"].nunique())

# =========================================================
# 2) SETUP
# =========================================================
def ce50_eleveld(age):
    return 3.08 * np.exp(-0.00635 * (age - 35))

bins = [18, 25, 35, 45, 55, 65, 75, 85, 90]
labels = ["18-24", "25-34", "35-44", "45-54", "55-64", "65-74", "75-84", "85-89"]

df_merged["Age_Group"] = pd.cut(df_merged["AGE"], bins=bins, labels=labels, right=False)

AGE_LO, AGE_HI = float(df_merged["AGE"].min()), float(df_merged["AGE"].max())

covariate_cols = [
    "ASA_SCORE", "BMI", "SEX",
    "TOTAL_FENT_MCG", "TOTAL_KETA_MG", "TOTAL_MIDA_MG", "TOTAL_ETOM_MG"
]

g_means = df_merged[covariate_cols].mean()

# Smooth prediction grid
age_grid = np.linspace(18, 89, 400)
grid_df = pd.DataFrame({"AGE": age_grid})
for col, val in g_means.items():
    grid_df[col] = val

# Binned summary ages
bin_stats = (
    df_merged.groupby("Age_Group", observed=True)
    .agg(mean_age=("AGE", "mean"))
    .reset_index()
)

bin_pred_df = pd.DataFrame({"AGE": bin_stats["mean_age"]})
for col, val in g_means.items():
    bin_pred_df[col] = val

# =========================================================
# 3) FIT MODELS
# =========================================================
formula_ce = (
    f"Max_Effect_Concentration ~ bs(AGE, df=5, lower_bound={AGE_LO}, upper_bound={AGE_HI}) + "
    "ASA_SCORE + BMI + SEX + TOTAL_FENT_MCG + TOTAL_KETA_MG + TOTAL_MIDA_MG + TOTAL_ETOM_MG"
)

formula_dose = (
    f"DOSE_PER_ABW ~ bs(AGE, df=5, lower_bound={AGE_LO}, upper_bound={AGE_HI}) + "
    "ASA_SCORE + BMI + SEX + TOTAL_FENT_MCG + TOTAL_KETA_MG + TOTAL_MIDA_MG + TOTAL_ETOM_MG"
)

model_ce = smf.ols(formula_ce, data=df_merged).fit()
model_dose = smf.ols(formula_dose, data=df_merged).fit()

# Predictions
pred_grid_ce = model_ce.get_prediction(grid_df).summary_frame()
pred_grid_dose = model_dose.get_prediction(grid_df).summary_frame()

bin_pred_ce = model_ce.get_prediction(bin_pred_df).summary_frame()
bin_pred_dose = model_dose.get_prediction(bin_pred_df).summary_frame()

# =========================================================
# 4) NORMALIZED DIVERGENCE CALCULATIONS
# =========================================================
target_ce50 = ce50_eleveld(bin_stats["mean_age"])

# Normalize to youngest cohort
baseline_ce = bin_pred_ce["mean"].iloc[0]
baseline_dose = bin_pred_dose["mean"].iloc[0]
baseline_target = target_ce50.iloc[0]

bin_stats["Dose_Pct"] = (bin_pred_dose["mean"] / baseline_dose) * 100
bin_stats["Ce_Pct"] = (bin_pred_ce["mean"] / baseline_ce) * 100
bin_stats["Ce_L"] = (bin_pred_ce["mean_ci_lower"] / baseline_ce) * 100
bin_stats["Ce_U"] = (bin_pred_ce["mean_ci_upper"] / baseline_ce) * 100
bin_stats["Target_Pct"] = (target_ce50 / baseline_target) * 100

# =========================================================
# 5) PANEL A: EXPOSURE VS REQUIREMENT
# =========================================================
plt.figure(figsize=(15, 9))
plt.plot(age_grid, pred_grid_ce["mean"], linewidth=4, color="#1f77b4", label="Adj. Predicted Peak $C_e$")
plt.plot(age_grid, ce50_eleveld(age_grid), linestyle="--", linewidth=3, color="red", label="Eleveld $Ce_{50}$ Target")

plt.bar(
    bin_stats["mean_age"],
    bin_pred_ce["mean"],
    width=3.5,
    color="#5dade2",
    alpha=0.6,
    edgecolor="#1f77b4",
    yerr=[
        bin_pred_ce["mean"] - bin_pred_ce["mean_ci_lower"],
        bin_pred_ce["mean_ci_upper"] - bin_pred_ce["mean"]
    ],
    capsize=5
)

for i, row in bin_pred_ce.iterrows():
    target = ce50_eleveld(bin_stats["mean_age"].iloc[i])
    diff = ((row["mean"] / target) - 1) * 100
    plt.text(
        bin_stats["mean_age"].iloc[i], row["mean"] + 0.12,
        f"{row['mean']:.2f}", ha="center", fontweight="bold", color="#1f77b4"
    )
    plt.text(
        bin_stats["mean_age"].iloc[i], row["mean"] + 0.38,
        f"+{diff:.0f}%", ha="center", fontweight="bold", color="red"
    )

plt.xticks(bin_stats["mean_age"], labels)
plt.title(
    "Peak Exposure ($C_e$) vs. Target\n(Adjusted for ASA, BMI, Sex, and Adjunct Induction Medications)",
    fontsize=15, fontweight="bold", pad=40
)
plt.ylabel("Effect-Site Concentration (mcg/mL)")
plt.ylim(0, 5.5)
plt.legend(bbox_to_anchor=(1.04, 1), loc="upper left")
plt.tight_layout()
plt.show()

# =========================================================
# 6) PANEL B: DOSE TITRATION
# =========================================================
plt.figure(figsize=(15, 8))
plt.plot(age_grid, pred_grid_dose["mean"], linewidth=4, color="#2ca02c", label="Adj. Induction Dose")
plt.bar(
    bin_stats["mean_age"],
    bin_pred_dose["mean"],
    width=3.5,
    color="#a1d99b",
    alpha=0.6,
    edgecolor="#2ca02c",
    yerr=[
        bin_pred_dose["mean"] - bin_pred_dose["mean_ci_lower"],
        bin_pred_dose["mean_ci_upper"] - bin_pred_dose["mean"]
    ],
    capsize=5
)

for i, row in bin_pred_dose.iterrows():
    plt.text(
        bin_stats["mean_age"].iloc[i], row["mean"] + 0.08,
        f"{row['mean']:.2f}", ha="center", fontsize=11, fontweight="bold", color="#2ca02c"
    )

plt.xticks(bin_stats["mean_age"], labels)
plt.title(
    "Standardized Clinician Induction Dose Titration (Ages 18-89)\n"
    "(Adjusted for ASA, BMI, Sex, and Adjunct Induction Medications)",
    fontsize=15, fontweight="bold", pad=20
)
plt.ylabel("Induction Dose (mg/kg ABW)")
plt.ylim(0, max(bin_pred_dose["mean_ci_upper"]) + 0.5)
plt.legend(bbox_to_anchor=(1.04, 1), loc="upper left")
plt.tight_layout()
plt.show()

# =========================================================
# 7) PANEL C: NORMALIZED DIVERGENCE
# =========================================================
plt.figure(figsize=(14, 8))
x = np.arange(len(labels))

plt.plot(
    x, bin_stats["Dose_Pct"], marker="o", linewidth=4, color="#2ca02c",
    label="Clinician Titration (Induction $mg/kg$ % Baseline)"
)

plt.plot(
    x, bin_stats["Ce_Pct"], marker="s", linewidth=4, color="#1f77b4",
    label="Modeled Peak Exposure (Predicted $C_e$ % Baseline)"
)
plt.fill_between(x, bin_stats["Ce_L"], bin_stats["Ce_U"], color="#1f77b4", alpha=0.15)

plt.plot(
    x, bin_stats["Target_Pct"], marker="D", linewidth=3, linestyle="--", color="red",
    label="Validated Age-Adjusted PD Reference ($Ce_{50}$ % Baseline)"
)

plt.xticks(x, labels)
plt.ylabel("Percentage of Young Adult (18-24) Baseline (%)", fontsize=12)
plt.title(
    "Relative Divergence: Clinical Titration vs. Pharmacological Target\n"
    "(Standardized to Mean Covariates and 18-24 Baseline)",
    fontsize=14, fontweight="bold", pad=20
)
plt.ylim(50, 115)
plt.legend(loc="lower left", frameon=True, facecolor="white", framealpha=0.9)
plt.grid(axis="y", linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()

# =========================================================
# 8) COMBINED MASTER FIGURE
# =========================================================
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 19))
plt.subplots_adjust(hspace=0.32, top=0.95, bottom=0.08, left=0.1, right=0.95)

# --- PANEL A ---
ax1.plot(age_grid, pred_grid_ce["mean"], linewidth=4, color="#1f77b4", label="Adj. Predicted Peak $C_e$")
ax1.plot(age_grid, ce50_eleveld(age_grid), linestyle="--", linewidth=3, color="red", label="Eleveld $Ce_{50}$ Target")
ax1.bar(
    bin_stats["mean_age"],
    bin_pred_ce["mean"],
    width=3.5,
    color="#5dade2",
    alpha=0.6,
    edgecolor="#1f77b4",
    yerr=[
        bin_pred_ce["mean"] - bin_pred_ce["mean_ci_lower"],
        bin_pred_ce["mean_ci_upper"] - bin_pred_ce["mean"]
    ],
    capsize=5
)

for i, row in bin_pred_ce.iterrows():
    target = ce50_eleveld(bin_stats["mean_age"].iloc[i])
    diff = ((row["mean"] / target) - 1) * 100
    ax1.text(
        bin_stats["mean_age"].iloc[i], row["mean"] + 0.12,
        f"{row['mean']:.2f}", ha="center", fontweight="bold", color="#1f77b4"
    )
    ax1.text(
        bin_stats["mean_age"].iloc[i], row["mean"] + 0.38,
        f"+{diff:.0f}%", ha="center", fontweight="bold", color="red"
    )

ax1.set_title("A: Standardized Peak Exposure ($C_e$) vs. Validated Target", loc="left", fontsize=14, fontweight="bold")
ax1.set_ylabel("Effect-Site Concentration (mcg/mL)")
ax1.set_ylim(0, 5.5)
ax1.set_xticks(bin_stats["mean_age"])
ax1.set_xticklabels(labels)
ax1.legend(loc="upper right", frameon=True)

# --- PANEL B ---
ax2.plot(age_grid, pred_grid_dose["mean"], linewidth=4, color="#2ca02c", label="Adj. Induction Dose")
ax2.bar(
    bin_stats["mean_age"],
    bin_pred_dose["mean"],
    width=3.5,
    color="#a1d99b",
    alpha=0.6,
    edgecolor="#2ca02c",
    yerr=[
        bin_pred_dose["mean"] - bin_pred_dose["mean_ci_lower"],
        bin_pred_dose["mean_ci_upper"] - bin_pred_dose["mean"]
    ],
    capsize=5
)

for i, row in bin_pred_dose.iterrows():
    ax2.text(
        bin_stats["mean_age"].iloc[i], row["mean"] + 0.08,
        f"{row['mean']:.2f}", ha="center", fontweight="bold", color="#2ca02c"
    )

ax2.set_title("B: Standardized Clinician Induction Dose Titration", loc="left", fontsize=14, fontweight="bold")
ax2.set_ylabel("Induction Dose (mg/kg ABW)")
ax2.set_xticks(bin_stats["mean_age"])
ax2.set_xticklabels(labels)
ax2.legend(loc="upper right", frameon=True)

# --- PANEL C ---
x_indices = np.arange(len(labels))
ax3.plot(
    x_indices, bin_stats["Dose_Pct"], marker="o", linewidth=4, color="#2ca02c",
    label="Clinician Titration (mg/kg % Baseline)"
)
ax3.plot(
    x_indices, bin_stats["Ce_Pct"], marker="s", linewidth=4, color="#1f77b4",
    label="Modeled Peak Exposure ($C_e$ % Baseline)"
)
ax3.fill_between(x_indices, bin_stats["Ce_L"], bin_stats["Ce_U"], color="#1f77b4", alpha=0.15)
ax3.plot(
    x_indices, bin_stats["Target_Pct"], marker="D", linewidth=3, linestyle="--", color="red",
    label="Validated PD Reference ($Ce_{50}$ % Baseline)"
)

ax3.set_title("C: The Titration-Response Discrepancy: Relative Divergence from Requirement", loc="left", fontsize=14, fontweight="bold")
ax3.set_ylabel("Percentage of Baseline (%)")
ax3.set_xlabel("Age Cohort (Years)", fontsize=13, labelpad=10)
ax3.set_xticks(x_indices)
ax3.set_xticklabels(labels)
ax3.set_ylim(40, 115)
ax3.legend(loc="lower left", frameon=True)
ax3.grid(axis="y", linestyle="--", alpha=0.3)

plt.savefig("figure_overexposure_analysis_adjuncts.pdf", bbox_inches="tight", dpi=300)
print("✅ Figure saved: figure_overexposure_analysis_adjuncts.pdf")
plt.show()

In [ ]:
# =========================================================
# 9) TABLE GENERATION FOR ADJUNCT-ADJUSTED SENSITIVITY ANALYSIS
# =========================================================

# Recalculate targets and surplus
targets_val = ce50_eleveld(bin_stats["mean_age"])
ov_mean = ((bin_pred_ce["mean"] / targets_val) - 1) * 100
ov_low = ((bin_pred_ce["mean_ci_lower"] / targets_val) - 1) * 100
ov_high = ((bin_pred_ce["mean_ci_upper"] / targets_val) - 1) * 100

# N per age group from the regression dataset actually used
group_counts = (
    df_merged.groupby("Age_Group", observed=True)
    .size()
    .reindex(labels, fill_value=0)
    .values
)

# Build table
table_adjuncts = pd.DataFrame({
    "Age Group": labels,
    "N": [f"{int(x):,}" for x in group_counts],
    "Induction Dose": [
        f"{m:.2f} ({l:.2f}--{u:.2f})"
        for m, l, u in zip(
            bin_pred_dose["mean"],
            bin_pred_dose["mean_ci_lower"],
            bin_pred_dose["mean_ci_upper"]
        )
    ],
    "Peak Ce (95% CI)": [
        f"{m:.2f} ({l:.2f}--{u:.2f})"
        for m, l, u in zip(
            bin_pred_ce["mean"],
            bin_pred_ce["mean_ci_lower"],
            bin_pred_ce["mean_ci_upper"]
        )
    ],
    "Surplus (95% CI)": [
        f"{m:.1f}% ({l:.1f}--{u:.1f}%)"
        for m, l, u in zip(ov_mean, ov_low, ov_high)
    ]
})

# Preview
print(table_adjuncts)

# Save to CSV
table_adjuncts.to_csv("table_sensitivity_adjuncts.csv", index=False)

# Export to LaTeX
with open("table_sensitivity_adjuncts.tex", "w") as f:
    f.write(
        table_adjuncts.to_latex(
            index=False,
            escape=False,
            column_format="lcccc"
        ).replace("%", r"\%")
    )

print("✅ Success: table_sensitivity_adjuncts.csv and table_sensitivity_adjuncts.tex saved.")


## Combine sensitivity-analysis tables for supplementary output

In [ ]:


# 1. Load sensitivity-analysis tables
df_sch = pd.read_csv("table_S1_schnider.csv")
df_raw = pd.read_csv("table_sensitivity_raw.csv")
df_out = pd.read_csv("table_sensitivity_outliers.csv")
df_90p = pd.read_csv("table_sensitivity_90plus.csv")
df_meds = pd.read_csv("table_sensitivity_adjuncts.csv")

# 2. Standardize age-group labels
def standardize_age(label):
    if str(label) in ["85-89", "85+", "85-90", "85+*"]:
        return "85+*"
    return label

for df in [df_sch, df_raw, df_out, df_90p, df_meds]:
    col = next(c for c in df.columns if "Age" in c)
    df.rename(columns={col: "Age Group"}, inplace=True)
    df["Age Group"] = df["Age Group"].apply(standardize_age)

# 3. Helper function
def prep_subtable(df, model_label):
    cols_to_keep = {
        "Age Group": "Age Cohort",
        "N": "N",
        "Induction Dose": "Dose (mg/kg)",
        "Peak Ce (95% CI)": "Peak $C_e$"
    }

    # Schnider table may use a different peak-Ce column name
    if "Schnider" in model_label and "Peak Ce" in df.columns and "Peak Ce (95% CI)" not in df.columns:
        df = df.rename(columns={"Peak Ce": "Peak Ce (95% CI)"})

    sub = df[[c for c in df.columns if c in cols_to_keep.keys()]].copy()
    sub.rename(columns=cols_to_keep, inplace=True)

    # Surplus column
    if "Schnider" in model_label:
        sub["Surplus %"] = "---"
    else:
        sub["Surplus %"] = df["Surplus (95% CI)"]

    # Section header row
    header_row = pd.DataFrame({
        "Age Cohort": [f"\\textbf{{{model_label}}}"],
        "N": [""],
        "Dose (mg/kg)": [""],
        "Peak $C_e$": [""],
        "Surplus %": [""]
    })

    return pd.concat([header_row, sub], ignore_index=True)

# 4. Process sections in desired order
t1 = prep_subtable(df_sch, "Schnider PK Model")
t2 = prep_subtable(df_raw, "Arithmetic (Raw) Analysis")
t3 = prep_subtable(df_out, "Broader-Input Cohort")
t4 = prep_subtable(df_90p, "Age 90+ Extended Analysis")
t5 = prep_subtable(df_meds, "Adjustment for Co-administered Induction Medications")

# 5. Stack and export
master_stacked = pd.concat([t1, t2, t3, t4, t5], ignore_index=True)

latex_table = master_stacked.to_latex(index=False, escape=False, column_format="lcccc")
final_latex = latex_table.replace("%", r"\%").replace(r"\\%", r"\%")

with open("table_master_sensitivity.tex", "w") as f:
    f.write(final_latex)

print("✅ Stacked sensitivity table saved with Schnider first and medication-adjusted model included.")